# Neural-network binary mask 

This notebook generates binary cell masks from fluorescence microscopy images using the pretrained U-Net segmentation model. The workflow is designed for low signal-to-noise fluorescence datasets in which conventional thresholding or classical segmentation approaches often fail to accurately detect dim cellular structures.

The script loads either a single image or an entire folder of images, applies the pretrained neural network, and outputs binary segmentation masks. The resulting masks can then be used for downstream watershed separation, FRET analysis, ratiometric quantification, or additional image-processing pipelines.

A pretrained model is provided, but the workflow can also be adapted to custom-trained models for different reporters, microscopes, or imaging conditions.




### Required packages

Required packages can be installed with:

```bash
pip install numpy opencv-python matplotlib tifffile torch segmentation-models-pytorch scikit-image
```

The workflow can run on CPU-only systems and does not require a dedicated GPU. If available, the script automatically detects and uses Apple Metal (`mps`) or CUDA acceleration.

In [14]:
import os
import glob
import cv2
import numpy as np
import torch
from tifffile import imwrite
from segmentation_models_pytorch import Unet

### Define input/output paths

Set the path to the input image or image folder, the pretrained neural-network model, and the output folder where the predicted binary masks will be saved.
The output directory is automatically created if it does not already exist.

In [15]:

INPUT_IMAGE = "example_image&mask/raw_image.tif"
MODEL_PATH = "best_semantic_model.pth"
OUT_DIR = "example_image&mask"

os.makedirs(OUT_DIR, exist_ok=True)

### Model settings and device selection

This section defines the neural-network encoder and the predicted threshold used to generate the binary mask.
The script automatically detects the available hardware and runs using Apple Metal (`mps`), CUDA, or CPU.

In [16]:
ENCODER_NAME = "resnet50"
THRESHOLD = 0.75

device = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Using device:", device)

Using device: mps


### Load pretrained segmentation model

This step loads the pretrained U-Net segmentation model and prepares it for inference on the selected device.

In [17]:
model = Unet(
    encoder_name=ENCODER_NAME,
    in_channels=3,
    classes=1
)

ckpt = torch.load(MODEL_PATH, map_location=device, weights_only=False)

if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
    model.load_state_dict(ckpt["model_state_dict"])
else:
    model.load_state_dict(ckpt)

model.to(device).eval();

### Helper functions

These helper functions load the input images, convert them to grayscale if needed, preprocess them for the neural network, and generate the final binary segmentation masks from the model predictions.

In [18]:

def list_images(path):
    if os.path.isfile(path):
        return [path]

    files = []
    for ext in ("*.tif", "*.tiff", "*.png", "*.jpg", "*.jpeg"):
        files.extend(glob.glob(os.path.join(path, ext)))

    return sorted(files)


def read_image(path):
    img = cv2.imread(path, cv2.IMREAD_UNCHANGED)

    if img is None:
        raise ValueError(f"Could not read image: {path}")

    if img.ndim == 3:
        if img.shape[2] == 4:
            img = img[:, :, :3]
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    return img


def predict_binary_mask(img):
    img = img.astype(np.float32) / 255.0
    img = (img - 0.5) / 0.5
    img_rgb = np.repeat(img[..., None], 3, axis=2)

    x = torch.from_numpy(img_rgb.transpose(2, 0, 1)).unsqueeze(0).float().to(device)

    with torch.no_grad():
        pred = model(x)

        if isinstance(pred, tuple):
            pred = pred[0]

        prob = torch.sigmoid(pred)[0, 0].cpu().numpy()

    mask = (prob > THRESHOLD).astype(np.uint8)

    return mask

### Run neural-network segmentation

This step runs the pretrained neural network on all input images and saves one binary segmentation mask for each image in the output directory.

In [19]:
files = list_images(INPUT_IMAGE)
print(f"Found {len(files)} image(s)")

for path in files:
    name = os.path.splitext(os.path.basename(path))[0]

    raw = read_image(path)
    mask = predict_binary_mask(raw)

    out_path = os.path.join(OUT_DIR, f"NN_binary_mask.tif")

    imwrite(out_path, (mask * 255).astype(np.uint8))

    print(f"Saved binary mask: {out_path}")

print("Done.")

Found 1 image(s)
Saved binary mask: example_image&mask/NN_binary_mask.tif
Done.
